In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import json
from fABBA import JABBA
from sklearn import preprocessing
from sklearn.metrics import matthews_corrcoef
from sklearn.metrics import accuracy_score
from src.preprocessing import encoders, vector_embed
import torch
import math
from datasets.dataset_dict import DatasetDict
from datasets import Dataset    
from transformers import AutoModelForSequenceClassification
from peft import get_peft_model, LoraConfig, TaskType

import evaluate
import numpy as np
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, DataCollatorWithPadding

import warnings

warnings.filterwarnings("ignore")

torch.cuda.empty_cache()

roberta_checkpoint = "roberta-large"
mistral_checkpoint = 'mistralai/Mistral-7B-Instruct-v0.1'
llama_checkpoint = "meta-llama/Llama-2-7b-hf"


2024-05-14 22:00:54.671368: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-14 22:01:08.880970: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


# Load data from the UCR dataset

In [5]:
# Python program to convert a list to string
def listToString(s):
    # initialize an empty string
    str1 = ""
    # traverse in the string
    for ele in s:
        str1 += ele + ' '
    # return string
    return str1


current_dir = "UCR2018"
UCR_folder = os.listdir(current_dir)
output_dir = 'UCRs'
try:
    UCR_folder.remove("Missing_value_and_variable_length_datasets_adjusted")
    UCR_folder.remove(".ipynb_checkpoints")
except:
    pass


directory_all = sorted(UCR_folder)[2:]


for file_i in range(1):#len(directory_all)):

    directory = directory_all[file_i]  ## 57: Haptics  41: FordA  33: Earthquakes  24: DistalPhalanxTW
    print(directory)
    
    input_folder = current_dir + "/" + directory
    output_folder = output_dir + "/" + directory
    if not os.path.exists(output_folder):
        os.mkdir(output_dir + "/" + directory) # create the file for saving the LORA Block

    # data reading
    train = pd.read_csv(input_folder + '/' + directory + '_TRAIN.tsv', sep='\t', header=None)
    test = pd.read_csv(input_folder + '/' + directory + '_TEST.tsv', sep='\t', header=None)

    # data discription
    with open(input_folder + '/' + 'README.md', 'r') as file:
        num_line = 0
        for line in file:
            if num_line == 2:
                data_disctiption = line
                break
            num_line += 1


    train_X = train[train.columns[1:]]
    test_X = test[test.columns[1:]]

    # train_X = (train_X - train_X.mean(axis=0)) /train_X.std(axis=0)
    # test_X = (test_X - test_X.mean(axis=0)) /test_X.std(axis=0)

    split = train.shape[0] 
    print("<- There are " + str(len(np.unique(train[0].values))) + " classes!")
    num_classes = 11 # len(np.unique(train[0].values))

    train_targets = train[0].values
    val_targets = train_targets[:int(split*0.2)]
    test_targets = test[0].values

    
    label_weights = torch.ones(num_classes)
    
    data = pd.concat([train_X, test_X])

    qabba = JABBA(tol=0.0001, init="agg", k=10000, scl=3, verbose=0)
    ## init kmeans or agg
    ## tol： compress rate, lower is better
    ## k： bigger is better
    ## scl： vabration rate, bigger is 序列对齐程度
    ## verbose： info 
    symbols = qabba.fit_transform(data)
    reconst = qabba.inverse_transform(symbols) #  set a shreshold that can optimize the generation
    
    plt.plot(train[train.columns[1]].values, label='time series')
    plt.plot(reconst, label='reconstruction')
    plt.legend()
    plt.grid(True, axis='y')
    plt.show()

    symbols_convert = []
    for i_convert in range(len(symbols)):

        symbols_convert.append(listToString(symbols[i_convert]))


    symbols_train = symbols_convert[:split]
    symbols_val = symbols_train[:int(split*0.2)]
    symbols_test = symbols_convert[split:]

    MAX_LEN = len(symbols_train[0])    #  pow(2, int(math.log2(len(symbols_train[0]))))
    print("MAX_LEN is:" + str(MAX_LEN))

    print("The length of Training input is:" + str(len(symbols_train[0])))
#     print("Testing input is:" + symbols_test[0])

    data_UCR = []
    data_UCR = DatasetDict({
        'train':Dataset.from_dict({'label':train_targets,'text':symbols_train}),
        'val':Dataset.from_dict({'label':val_targets,'text':symbols_val}),
        'test':Dataset.from_dict({'label':test_targets,'text':symbols_test})
         })


AllGestureWiimoteX
<- There are 10 classes!
MAX_LEN is:232
The length of Training input is:232
AllGestureWiimoteY
<- There are 10 classes!
MAX_LEN is:82
The length of Training input is:82


In [24]:
len(reconst[1])

500

In [33]:
train[train.iterrows].values.shape

TypeError: DataFrame.iterrows() takes 1 positional argument but 2 were given

In [39]:
train[1].values.shape

(300,)

In [37]:
train[train.columns[1:]]


,1,2,3,4,5,6,7,8,9,10,...,491,492,493,494,495,496,497,498,499,500
0,0.074,0.037,0.037,0.000,0.074,0.074,0.148,0.222,0.222,0.222,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.148,0.148,0.222,0.111,0.037,0.000,0.074,0.000,0.000,0.074,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.074,0.111,0.111,0.148,0.074,0.037,0.074,0.111,0.222,0.222,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-0.037,0.000,0.000,0.000,0.000,-0.037,-0.037,0.037,0.037,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.037,0.000,0.000,0.074,-0.037,0.000,0.111,0.074,0.074,0.111,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,0.037,-0.037,0.074,0.111,0.259,0.259,0.333,0.407,0.444,0.481,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
296,-0.074,-0.037,-0.037,-0.037,-0.037,-0.037,0.000,-0.037,-0.037,-0.037,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
297,0.000,-0.037,-0.037,-0.037,-0.037,-0.037,0.000,0.000,-0.037,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
298,0.037,-0.111,-0.074,-0.037,-0.074,-0.074,-0.074,-0.074,-0.074,-0.074,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
train_X = train[train.columns[1:]]
test_X = test[test.columns[1:]]

# train_X = (train_X - train_X.mean(axis=0)) /train_X.std(axis=0)
# test_X = (test_X - test_X.mean(axis=0)) /test_X.std(axis=0)

train_targets = train[0].values
test_targets = test[0].values

split = train.shape[0]
data = pd.concat([train_X, test_X])
# data.fillna(0, inplace=True)

In [5]:
print(train.shape)
print(train_X.shape)

(100, 1461)
(100, 1460)


## **************************************************************************************

## Part I: Convert numerical to symbolic varaibles using ABBA 

In the first part, we believe that those pre-trained Large Lanuage Models (LLMs) know the symbols generated from the ABBA method. The preconditon is that LLMs can process these symbols and understand their information that hide behind this symbolic time series. 

In [6]:
# data = (data - data.mean(axis=0)) / data.std(axis=0)
data.shape

(200, 1460)

In [7]:
qabba = JABBA(tol=0.005, init="kmeans", k=250, scl=3, verbose=0)
## init kmeans or agg
## tol： compress rate, lower is better
## k： bigger is better
## scl： vabration rate, bigger is 序列对齐程度
## verbose： info 
symbols = qabba.fit_transform(data)
reconst = qabba.inverse_transform(symbols) #  set a shreshold that can optimize the generation


In [8]:
symbols_train = symbols[:split]
symbols_test = symbols[split:]

### 1. Prepear the Instruction Text Data

In [9]:
data_template = {
    "text": [],
    "labels": [],
}

In [10]:

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

train_data_symbolic = []
test_data_symbolic = []
# Saving training data to JSON file
for train_i in range(len(symbols_train)):
    data_template["text"] = symbols_train[train_i]
    data_template["labels"] = train_targets[train_i]
    train_data_symbolic.append(data_template)
f_json = folder + '/' + directory + '_TRAIN.json'
with open(f_json, 'w', encoding='utf8') as f:
    json.dump(train_data_symbolic, f, ensure_ascii=False, cls=NpEncoder)
f.close()


# Saving testing data to JSON file
for test_i in range(len(symbols_test)):
    data_template["text"] = symbols_test[test_i]
    data_template["labels"] = test_targets[test_i]
    test_data_symbolic.append(data_template)
f_json = folder + '/' + directory + '_TEST.json'
with open(f_json, 'w', encoding='utf8') as f:
    json.dump(test_data_symbolic, f, ensure_ascii=False, cls=NpEncoder)
f.close()

# data_symbolic = {
#     "train": [],
#     "test": [],
# }
# data_symbolic['train'] = train_data_symbolic
# data_symbolic['test'] = test_data_symbolic

# f_json = folder + '/' + directory + '.json'
# with open(f_json, 'w', encoding='utf8') as f:
#     json.dump(data_symbolic, f, ensure_ascii=False, cls=NpEncoder)
# f.close()

In [11]:
data_template

{'series': ['ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  'D',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  'D',
  'ċ',
  'ĕ',
  'î',
  'D',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'V',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  '°',
  'ċ',
  'ĕ',
  'î',
  'D',
  'ċ',
  'ĕ',
  'î',
  'D',
  'ċ',
  'ĕ',
  'î',
  'D',
  'ċ',
  'ĕ',
  'î',
  'D',
  'ċ',
  'ĕ',
  'î',
  'D',
  'ċ',
  'ĕ',
  'î',
  'D',
  'ċ',
  'ĕ',
  'î',
  'D',
  'ċ',
  'ĕ',
  'î',
  'D',
  'ċ',
  'ĕ',
  '

### 2. Load Large Language Models (e.g. Mistreal-7B)

As this is a classification task, the input <font face="Black" color="Blue" size="3">Data Format</font> should be:

```ruby
data_template = {
{'text': ['ĕ', 'î', ... '¤', '°', 'd', ...],
 'labels': 1
}

```

When using pre-trained LLMs, we should carfully set the <font face="Black" color="Blue" size="3">Prompt</font> template.

```ruby
prompt_template = {
    "prompt": (
        "Below is an instruction that describes a task, paired with an input that provides further context. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:"
    ),
    "disctiption": "The dataset is compiled from ACS-F1, the first version of the database of appliance consumption signatures. The dataset contains the power consumption of typical appliances. The recordings are characterized by long idle periods and some high bursts of enery consumption when the appliance is active.\n",
    "instruction": "Given a symbolic series, predict the category.",
    "task": "classification",
    "text": ['ĕ', 'î', ... '¤', '°', 'd', ...],
    "labels": 1
}
```



In [2]:

import torch
from peft import LoraConfig, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline
)
import functools


#################################################################
# Tokenizer
#################################################################
model_name = 'mistralai/Mistral-7B-Instruct-v0.1'
# model_name = 'mistralai/Mistral-7B-v0.1'

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

#################################################################
# bitsandbytes parameters
#################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False

#################################################################
# Set up quantization config
#################################################################
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
# if compute_dtype == torch.float16 and use_4bit:
#     major, _ = torch.cuda.get_device_capability()
#     if major >= 8:
#         print("=" * 80)
#         print("Your GPU supports bfloat16: accelerate training with bf16=True")
#         print("=" * 80)

#################################################################
# Load pre-trained config
#################################################################
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
)

# model = AutoModelForSequenceClassification.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     num_labels=1
# )


def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(model))




Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at mistralai/Mistral-7B-Instruct-v0.1 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable model parameters: 131342336
all model parameters: 3621003264
percentage of trainable model parameters: 3.63%


In [13]:
def tokenize(prompt):
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

And convert each sample into a prompt that I found from this [notebook](https://github.com/samlhuillier/viggo-finetune/blob/main/llama/fine-tune-code-llama.ipynb).

In [14]:

def generate_and_tokenize_prompt(data_point):
    
#     full_prompt =f"""Given a target sentence construct the underlying meaning representation of the input sentence as a single function with attributes and attribute values.
# This function should describe the target string accurately and the function must be one of the following ['inform', 'request', 'give_opinion', 'confirm', 'verify_attribute', 'suggest', 'request_explanation', 'recommend', 'request_attribute'].
# The attributes must be one of the following: ['name', 'exp_release_date', 'release_year', 'developer', 'esrb', 'rating', 'genres', 'player_perspective', 'has_multiplayer', 'platforms', 'available_on_steam', 'has_linux_release', 'has_mac_release', 'specifier']

# ### Target sentence:
# {data_point["series"]}

# ### Meaning representation:
# {data_point["classification"]}
#     """
    
    full_prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context.
        Write a response that appropriately completes the request.
        
        ### Instruction:
        Given a symbolic series, predict its category. The category must be one of numbers: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
        
        ### Series:
        {data_point["text"]}
        
        ### Category:
        {data_point["labels"]}
        """
   
    return tokenize(full_prompt)


In [15]:
!pip show requests urllib3

Name: requests
Version: 2.28.2
Summary: Python HTTP for Humans.
Home-page: https://requests.readthedocs.io
Author: Kenneth Reitz
Author-email: me@kennethreitz.org
License: Apache 2.0
Location: /home/kangchen/.local/lib/python3.11/site-packages
Requires: certifi, charset-normalizer, idna, urllib3
Required-by: datasets, fABBA, huggingface-hub, torchvision, transformers, wandb
---
Name: urllib3
Version: 1.26.18
Summary: HTTP library with thread-safe connection pooling, file post, and more.
Home-page: https://urllib3.readthedocs.io/
Author: Andrey Petrov
Author-email: andrey.petrov@shazow.net
License: MIT
Location: /home/kangchen/.local/lib/python3.11/site-packages
Requires: 
Required-by: requests, sentry-sdk


In [16]:
!pip install --user requests==2.28.2 urllib3==1.26.18
!pip install --user datasets==2.16.0


[notice] A new release of pip is available: 23.1.2 -> 23.3.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.1.2 -> 23.3.2
[notice] To update, run: pip install --upgrade pip


Reformat the prompt and tokenize each sample:

In [15]:
from datasets import load_dataset

train_data_loader = load_dataset("json", data_files=folder + '/' + directory + '_TRAIN.json')
tokenized_train_dataset = (
    train_data_loader["train"].map(generate_and_tokenize_prompt)
)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [16]:
val_data_loader = load_dataset("json", data_files=folder + '/' + directory + '_TEST.json')
tokenized_val_dataset  = (
    val_data_loader["train"].map(generate_and_tokenize_prompt)
)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Check that `input_ids` is padded on the left with the `eos_token` (2) and there is an `eos_token` 2 added to the end, and the prompt starts with a `bos_token` (1).

In [17]:
print(tokenized_train_dataset[4]['input_ids'])
print(len(tokenized_train_dataset[4]['input_ids']))

[1, 20811, 349, 396, 13126, 369, 13966, 264, 3638, 28725, 5881, 1360, 395, 396, 2787, 369, 5312, 3629, 2758, 28723, 13, 5390, 12018, 264, 2899, 369, 6582, 1999, 2691, 274, 272, 2159, 28723, 13, 273, 13, 5390, 774, 3133, 3112, 28747, 13, 5390, 12628, 264, 5602, 294, 3518, 28725, 6782, 871, 8011, 28723, 415, 8011, 1580, 347, 624, 302, 5551, 28747, 5936, 28734, 647, 464, 28740, 647, 464, 28750, 647, 464, 28770, 647, 464, 28781, 647, 464, 28782, 647, 464, 28784, 647, 464, 28787, 647, 464, 28783, 647, 464, 28774, 1421, 13, 273, 13, 5390, 774, 10264, 28747, 13, 5390, 5936, 29656, 647, 464, 30305, 647, 464, 28793, 647, 464, 28796, 647, 464, 28964, 647, 464, 30452, 647, 464, 28766, 647, 464, 28824, 647, 464, 28943, 647, 464, 28793, 647, 464, 31619, 647, 464, 28936, 647, 464, 28854, 647, 464, 28793, 647, 464, 28806, 647, 464, 28758, 647, 464, 29198, 647, 464, 28793, 647, 464, 28715, 647, 464, 28712, 647, 464, 29198, 647, 464, 28793, 647, 464, 28806, 647, 464, 28758, 647, 464, 29198, 647, 464, 2

### 3. Set Up LoRA

Now, to start our fine-tuning, we have to apply some preprocessing to the model to prepare it for training. For that use the `prepare_model_for_kbit_training` method from PEFT.

In [18]:
from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [19]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

Let's print the model to examine its layers, as we will apply <font face="Black" color="Blue" size="3">QLoRA</font> to all the linear layers of the model. Those layers are `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`, and `lm_head`.

Here we define the LoRA config:

`r` is the rank of the low-rank matrix used in the adapters, which thus controls the number of parameters trained. A higher rank will allow for more expressivity, but there is a compute tradeoff.

`alpha` is the scaling factor for the learned weights. The weight matrix is scaled by `alpha/r`, and thus a higher value for alpha assigns more weight to the LoRA activations.

The values used in the QLoRA paper were `r=64` and `lora_alpha=16`, and these are said to generalize well, but we will use `r=8` and `lora_alpha=16` so that we have more emphasis on the new fine-tuned data while also reducing computational complexity.

In [20]:
from peft import LoraConfig, get_peft_model
from accelerate import FullyShardedDataParallelPlugin, Accelerator
from torch.distributed.fsdp.fully_sharded_data_parallel import FullOptimStateDictConfig, FullStateDictConfig

fsdp_plugin = FullyShardedDataParallelPlugin(
    state_dict_config=FullStateDictConfig(offload_to_cpu=True, rank0_only=False),
    optim_state_dict_config=FullOptimStateDictConfig(offload_to_cpu=True, rank0_only=False),
)

accelerator = Accelerator(fsdp_plugin=fsdp_plugin)

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        "lm_head",
    ],
    bias="none",
    lora_dropout=0.05,  # Conventional
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, config)
print_trainable_parameters(model)

# Apply the accelerator. You can comment this out to remove the accelerator.
model = accelerator.prepare_model(model)

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


trainable params: 21260288 || all params: 3773331456 || trainable%: 0.5634354746703705


### 4. Run Training

I used 500 steps, but I found the model should have trained for longer as it had not converged by then, so I upped the steps to 1000 below.

A note on training. You can set the `max_steps` to be high initially, and examine at what step your model's performance starts to degrade. There is where you'll find a sweet spot for how many steps to perform. For example, say you start with 1000 steps, and find that at around 500 steps the model starts overfitting - the validation loss goes up (bad) while the training loss goes down significantly, meaning the model is learning the training set really well, but is unable to generalize to new datapoints. Therefore, 500 steps would be your sweet spot, so you would use the `checkpoint-500` model repo in your output dir (`mistral-finetune-abba`) as your final model in step 6 below.

You can interrupt the process via Kernel -> Interrupt Kernel in the top nav bar once you realize you didn't need to train anymore.

In [21]:
import wandb, os
wandb.login()

wandb_project = "abba-finetune"
if len(wandb_project) > 0:
    os.environ["WANDB_PROJECT"] = wandb_project

wandb: Currently logged in as: chengkang520 (cheng_nlp). Use `wandb login --relogin` to force relogin


In [22]:
if torch.cuda.device_count() > 1: # If more than 1 GPU
    model.is_parallelizable = True
    model.model_parallel = True

In [24]:
import transformers
from datetime import datetime

project = "abba-finetune"
base_model_name = "mistral"
run_name = base_model_name + "-" + project
output_dir = "./" + run_name

tokenizer.pad_token = tokenizer.eos_token

trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    args=transformers.TrainingArguments(
        output_dir=output_dir,
        warmup_steps=5,
        per_device_train_batch_size=2,
        gradient_checkpointing=True,
        gradient_accumulation_steps=4,
        max_steps=200,
        learning_rate=2.5e-4, # Want about 10x smaller than the Mistral learning rate
        logging_steps=100,
        bf16=True,
        optim="paged_adamw_8bit",
        logging_dir="./logs",        # Directory for storing logs
        save_strategy="steps",       # Save the model checkpoint every logging step
        save_steps=100,                # Save checkpoints every 50 steps
        evaluation_strategy="steps", # Evaluate the model every logging step
        eval_steps=100,               # Evaluate and save checkpoints every 50 steps
        do_eval=True,                # Perform evaluation at the end of training
        report_to="wandb",           # Comment this out if you don't want to use weights & baises
        run_name=f"{run_name}-{datetime.now().strftime('%Y-%m-%d-%H-%M')}"          # Name of the W&B run (optional)
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
trainer.train()

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss
100,0.020200,0.729050
200,0.000000,0.770084


Checkpoint destination directory ./mistral-abba-finetune/checkpoint-100 already exists and is non-empty.Saving will proceed but saved results may be invalid.
/home/kangchen/.local/lib/python3.11/site-packages/peft/utils/save_and_load.py:134: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/home/kangchen/.local/lib/python3.11/site-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
Checkpoint destination directory ./mistral-abba-finetune/checkpoint-200 already ex

TrainOutput(global_step=200, training_loss=0.010078222262236522, metrics={'train_runtime': 655.1199, 'train_samples_per_second': 2.442, 'train_steps_per_second': 0.305, 'total_flos': 3.50548150714368e+16, 'train_loss': 0.010078222262236522, 'epoch': 16.0})

### 5. Drum Roll... Try the Trained Model

It's a good idea to kill the current process so that you don't run out of memory loading the base model again on top of the model we just trained. Go to `Kernel > Restart Kernel` or kill the process via the Terminal (`nvidia smi` > `kill [PID]`).

By default, the PEFT library will only save the QLoRA adapters, so we need to first load the base Mistral model from the Huggingface Hub:

In [25]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = 'mistralai/Mistral-7B-Instruct-v0.1'

# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"


import torch
from peft import LoraConfig, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline
)

#################################################################
# Tokenizer
#################################################################
base_model_id = 'mistralai/Mistral-7B-Instruct-v0.1'

tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

#################################################################
# bitsandbytes parameters
#################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False

#################################################################
# Set up quantization config
#################################################################
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
# if compute_dtype == torch.float16 and use_4bit:
#     major, _ = torch.cuda.get_device_capability()
#     if major >= 8:
#         print("=" * 80)
#         print("Your GPU supports bfloat16: accelerate training with bf16=True")
#         print("=" * 80)

#################################################################
# Load pre-trained config
#################################################################
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
)

eval_tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    add_bos_token=True,
    trust_remote_code=True,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Now load the QLoRA adapter from the appropriate checkpoint directory, i.e. the best performing model checkpoint:

In [26]:
from peft import PeftModel

ft_model = PeftModel.from_pretrained(base_model, "mistral-abba-finetune/checkpoint-200")

and run your inference!

Let's try the same eval_prompt and thus model_input as above, and see if the new finetuned model performs better.

In [31]:
eval_prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context.
        Write a response that appropriately completes the request.
        
        ### Instruction:
        Given a symbolic series, predict its category. The category must be one of numbers: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
        
        ### Series:
        {train_data_symbolic[0]["text"]}
        
        ### Category:
        """

# Re-init the tokenizer so it doesn't add padding or eos token
eval_tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    add_bos_token=True,
)

model_input = eval_tokenizer(eval_prompt, return_tensors="pt").to("cuda")

ft_model.eval()
with torch.no_grad():
    print(eval_tokenizer.decode(ft_model.generate(**model_input, max_new_tokens=256)[0], skip_special_tokens=True))
    


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Below is an instruction that describes a task, paired with an input that provides further context.
        Write a response that appropriately completes the request.
        
        ### Instruction:
        Given a symbolic series, predict its category. The category must be one of numbers: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
        
        ### Series:
        ['ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', 'D', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', 'D', 'ċ', 'ĕ', 'î', 'D', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'V', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', '°', 'ċ', 'ĕ', 'î', 'D', 'ċ', 'ĕ', 'î', 'D', 'ċ', 'ĕ', 'î', 'D', 'ċ', 'ĕ', 'î', 'D', 'ċ',

In [34]:
print("The Category should be: " + str(test_data_symbolic[0]["classification"]))

The Category should be: 1


## **************************************************************************************

## Part II: Construct the embedding space from scratch (using ABBA)

In the second part, we manage to replace the embedding layer with the QABBA processed embedding layer. Therefore, we should pre-train language models from-scratch.

In [ ]:
# will use the ABBA embedding here, use the centers as embedding so without additional embedding layer training.
param = qabba.parameters # ABBA model parameters, including centers, and hash mapping
# param

In [ ]:
# the less the percent (\in (0, 1] it uses, the more truncation of the sentence will be)
hashm = dict(zip(param.alphabets, np.arange(len(param.alphabets))))
ec = encoders(dictionary=hashm) #  load the params, and use the hash mapping for categorical encoding
train_data = ec.categorical_encode(symbols_train) # transform the symbols to categorical interger.
test_data = ec.categorical_encode(symbols_test) # transform the symbols to categorical interger.


In [ ]:
length = int(0.9* np.max([len(i) for i in train_data]) )

In [ ]:
padding_train = ec.pad_sequence(train_data, maxlen=length, value='none', method = "post", truncating='post')
padding_test = ec.pad_sequence(test_data, maxlen=length, value='none', method = "post", truncating='post')

In [ ]:
vsm = vector_embed(param.centers, hashm, string=False)
xtrain = vsm.transform(padding_train, std=True)
xtest = vsm.transform(padding_test, std=True)

In [ ]:
xtrain.shape

In [ ]:
xtrain[1,:,]

In [ ]:
train_targets